In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)
import sys
sys.path.append("..") 

In [2]:
spark = (
    SparkSession.builder
    .appName("Real-Time Fraud Transaction Detection")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

c:\Users\admin\Desktop\data_science_projects\real_time_fraud_transaction_detection\dev_env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0


In [3]:
df = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "transaction-stream")
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load()
)

print("Reading data from Kafka topic 'transaction-stream'...")
print(df.show(5, truncate=False))

json_df = df.select(
    col("value").cast("string").alias("json")
)

print("Converting Kafka messages to JSON format...")
print(json_df.show(3, truncate=False))

Reading data from Kafka topic 'transaction-stream'...
+----+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [4]:
transaction_schema = StructType([
    StructField("trans_date_trans_time", StringType(), True),
    StructField("cc_num", StringType(), True),
    StructField("merchant", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amt", StringType(), True),
    StructField("first", StringType(), True),
    StructField("last", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("street", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("zip", StringType(), True),
    StructField("lat", StringType(), True),
    StructField("long", StringType(), True),
    StructField("city_pop", StringType(), True),
    StructField("job", StringType(), True),
    StructField("dob", StringType(), True),
    StructField("trans_num", StringType(), True),
    StructField("unix_time", StringType(), True),
    StructField("merch_lat", StringType(), True),
    StructField("merch_long", StringType(), True),
    StructField("is_fraud", StringType(), True),
    StructField("merch_zipcode", StringType(), True)
])

trns_df = (
    df
    .select(
        from_json(
            col("value").cast("string"),
            transaction_schema
        ).alias("data")
    )
    .select("data.*")
)

print(f"rows: {trns_df.count()}, columns: {len(trns_df.columns)}")
trns_df.show(5, truncate=False)

rows: 3793, columns: 23
+---------------------+----------------+----------------------------------+-------------+------+---------+-------+------+----------------------------+--------------+-----+-----+-------+---------+--------+---------------------------------+----------+--------------------------------+----------+---------+-----------+--------+-------------+
|trans_date_trans_time|cc_num          |merchant                          |category     |amt   |first    |last   |gender|street                      |city          |state|zip  |lat    |long     |city_pop|job                              |dob       |trans_num                       |unix_time |merch_lat|merch_long |is_fraud|merch_zipcode|
+---------------------+----------------+----------------------------------+-------------+------+---------+-------+------+----------------------------+--------------+-----+-----+-------+---------+--------+---------------------------------+----------+--------------------------------+----------+-----

In [5]:
from pyspark.sql import Window, functions as F

# Cache the Kafka batch so quality checks and output use the same offsets.
trns_df = trns_df.persist()
raw_row_count = trns_df.count()

cleaned_transactions_df = (
    trns_df
    .withColumn("trans_date_trans_time", F.to_timestamp("trans_date_trans_time", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("dob", F.to_date("dob", "yyyy-MM-dd"))
    .withColumn("amt", F.col("amt").cast("double"))
    .withColumn("lat", F.col("lat").cast("double"))
    .withColumn("long", F.col("long").cast("double"))
    .withColumn("merch_lat", F.col("merch_lat").cast("double"))
    .withColumn("merch_long", F.col("merch_long").cast("double"))
    .withColumn("city_pop", F.col("city_pop").cast("long"))
    .withColumn("unix_time", F.col("unix_time").cast("long"))
    .withColumn("is_fraud", F.col("is_fraud").cast("int"))
    .withColumn("trans_num", F.trim("trans_num"))
    .withColumn("category", F.lower(F.trim("category")))
    .filter(
        F.col("trans_date_trans_time").isNotNull()
        & F.col("trans_num").isNotNull()
        & (F.length("trans_num") > 0)
        & F.col("amt").isNotNull()
        & (F.col("amt") >= 0)
        & F.col("unix_time").isNotNull()
        & F.col("is_fraud").isin(0, 1)
    )
    .dropDuplicates(["trans_num"])
)

window_10m = (
    Window.partitionBy("cc_num")
    .orderBy("unix_time")
    .rangeBetween(-10 * 60, -1)
)
window_1h = (
    Window.partitionBy("cc_num")
    .orderBy("unix_time")
    .rangeBetween(-60 * 60, -1)
)

transactions_df = (
    cleaned_transactions_df
    .withColumn("transaction_hour", F.hour("trans_date_trans_time"))
    .withColumn("transaction_day_of_week", F.dayofweek("trans_date_trans_time"))
    .withColumn("is_weekend", F.dayofweek("trans_date_trans_time").isin(1, 7).cast("int"))
    .withColumn("amt_log1p", F.log1p("amt"))
    .withColumn(
        "distance_km",
        F.round(
            6371.0 * 2 * F.asin(
                F.sqrt(
                    F.least(
                        F.lit(1.0),
                        F.pow(
                            F.sin(F.radians(F.col("merch_lat") - F.col("lat")) / 2),
                            2,
                        )
                        + F.cos(F.radians("lat"))
                        * F.cos(F.radians("merch_lat"))
                        * F.pow(
                            F.sin(F.radians(F.col("merch_long") - F.col("long")) / 2),
                            2,
                        ),
                    )
                )
            ),
            3,
        ),
    )
    .withColumn("card_txn_count_10m", F.count("trans_num").over(window_10m))
    .withColumn("card_amount_sum_10m", F.coalesce(F.sum("amt").over(window_10m), F.lit(0.0)))
    .withColumn("card_txn_count_1h", F.count("trans_num").over(window_1h))
    .withColumn("card_amount_sum_1h", F.coalesce(F.sum("amt").over(window_1h), F.lit(0.0)))
)

processed_row_count = transactions_df.count()
print(f"Valid unique transactions: {processed_row_count:,} of {raw_row_count:,} Kafka records")
transactions_df.select(
    "trans_num",
    "trans_date_trans_time",
    "amt",
    "is_fraud",
    "distance_km",
    "card_txn_count_10m",
    "card_amount_sum_10m",
    "card_txn_count_1h",
    "card_amount_sum_1h",
).show(5, truncate=False)


Valid unique transactions: 3,283 of 3,796 Kafka records
+--------------------------------+---------------------+-----+--------+-----------+------------------+-------------------+-----------------+------------------+
|trans_num                       |trans_date_trans_time|amt  |is_fraud|distance_km|card_txn_count_10m|card_amount_sum_10m|card_txn_count_1h|card_amount_sum_1h|
+--------------------------------+---------------------+-----+--------+-----------+------------------+-------------------+-----------------+------------------+
|14eb123e2dc9745d0ca4c87a5469ffe4|2019-01-01 01:23:26  |5.83 |0       |25.997     |0                 |0.0                |0                |0.0               |
|0cd2aff599108e9181a408f2d131b7ea|2019-01-01 16:26:09  |47.58|0       |63.217     |0                 |0.0                |0                |0.0               |
|c216532c65ea4eab9188a9c4cd6bceb4|2019-01-01 18:51:06  |77.7 |0       |27.848     |0                 |0.0                |0                |0.0 

In [6]:
from pathlib import Path

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / "config.py").is_file()),
    Path.cwd(),
)
processed_data_path = project_root / "data" / "processed" / "transactions_parquet"

processing_summary_df = transactions_df.agg(
    F.count("*").alias("transaction_count"),
    F.sum("is_fraud").alias("fraud_count"),
    F.avg("is_fraud").alias("fraud_rate"),
    F.avg("amt").alias("average_amount"),
)

print(f"Kafka records read: {raw_row_count:,}")
print(f"Valid unique transactions written: {processed_row_count:,}")
print(f"Records excluded or deduplicated: {raw_row_count - processed_row_count:,}")
processing_summary_df.show(truncate=False)

(
    transactions_df.write
    .mode("overwrite")
    .parquet(processed_data_path.as_posix())
)

persisted_transactions_df = spark.read.parquet(processed_data_path.as_posix())
persisted_row_count = persisted_transactions_df.count()
assert persisted_row_count == processed_row_count, (
    f"Parque  t row-count mismatch: wrote {processed_row_count}, read {persisted_row_count}"
)
print(f"Verified Parquet output: {processed_data_path} ({persisted_row_count:,} rows)")


Kafka records read: 3,796
Valid unique transactions written: 3,283
Records excluded or deduplicated: 513
+-----------------+-----------+---------------------+-----------------+
|transaction_count|fraud_count|fraud_rate           |average_amount   |
+-----------------+-----------+---------------------+-----------------+
|3283             |6          |0.0018275967103259215|64.85426743831879|
+-----------------+-----------+---------------------+-----------------+

Verified Parquet output: c:\Users\admin\Desktop\data_science_projects\real_time_fraud_transaction_detection\data\processed\transactions_parquet (3,283 rows)
